# Sparkle V2 — Fase 3.4: Agregación temporal por ventanas

**Protocolo pre-registrado:** `Protocolo_Exp1_Agregacion_Temporal.md` (2026-07-29).
Todo lo que se ejecuta acá está definido ahí ANTES de ver ningún número agregado.

**Resumen del protocolo congelado:**
- Ventana primaria: **k = 5 clips consecutivos** (50 s), mismo participante y video, sin solapamiento.
- Etiqueta de ventana: **mayoría** de las 5 etiquetas binarias de clip.
- Predicción de ventana: **media de las 5 probabilidades**, umbral 0,50.
- Trivial por ventana: se **recalcula** sobre las ventanas (no se hereda el 0,5440 por clip).
- Modelo A (primario): artefacto entregado, **sin reentrenar**. Modelo B (secundario, descriptivo).
- Desarrollo: CV interna de 5 folds por participante. **Test oficial: UNA sola pasada**,
  detrás de la compuerta manual de la sección 5.

**Entorno:** Colab (no requiere GPU para el resultado primario) · Drive `MyDrive/SparkleV2` montado.

In [ ]:
# =====================================================================
# SETUP
# =====================================================================
import sys, os, json
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')
assert os.path.isdir(PROJECT_DIR), f'No existe {PROJECT_DIR}'

CACHE_NPZ  = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
MODELO_DIR = f'{PROJECT_DIR}/modelo_final'
OUT_DIR    = f'{PROJECT_DIR}/fase3_4_agregacion'
os.makedirs(OUT_DIR, exist_ok=True)

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ------- Parámetros CONGELADOS por el pre-registro (no tocar) -------
SEED            = 42          # el mismo de 3.3: reproduce el split interno
K_PRIMARIO      = 5           # ventana de 5 clips = 50 s
UMBRAL          = 0.5         # regla dura 2: no se toca
N_BOOT          = 5000
KS_SECUNDARIOS  = [1, 3, 5, 7, 9]   # solo CV interna, jamás Test
N_FRAMES, N_EMB_FRAMES, EMB_DIM = 60, 16, 128
DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']
print('Protocolo cargado: k=5, mayoría, media de probabilidades, umbral 0,50')

## 1. Datos — la misma caché alineada de 3.2b/3.3

Solo se necesitan las secuencias crudas, las etiquetas y los **ClipIDs**. El preprocesamiento
se ajusta por fold en la CV (sección 3) y con `preprocesamiento.npz` en Test (sección 5).

In [ ]:
assert os.path.exists(CACHE_NPZ), (
    f'No está la caché {CACHE_NPZ}. Generarla corriendo la celda de datos de 3.2b o 3.3.')
d = np.load(CACHE_NPZ, allow_pickle=True)
Xg_tr, Xe_tr, y3_tr, id_tr = d['Xg_tr'], d['Xe_tr'], d['y3_tr'], d['id_tr']
Xg_va, Xe_va, y3_va, id_va = d['Xg_va'], d['Xe_va'], d['y3_va'], d['id_va']
Xg_te, Xe_te, y3_te, id_te = d['Xg_te'], d['Xe_te'], d['y3_te'], d['id_te']
FEATS22 = [str(f) for f in d['feature_names']]
id_tr, id_va, id_te = (np.array([str(c) for c in a]) for a in (id_tr, id_va, id_te))

# Pool Train+Validation (desarrollo). Test queda aparte hasta la sección 5.
Xg_pool = np.concatenate([Xg_tr, Xg_va]); Xe_pool = np.concatenate([Xe_tr, Xe_va])
y3_pool = np.concatenate([y3_tr, y3_va]); id_pool = np.concatenate([id_tr, id_va])
y_pool  = (y3_pool > 0).astype(np.int64)
part_pool = np.array([c[:6] for c in id_pool])
print(f'Pool {len(y_pool)} clips, {len(set(part_pool))} participantes | Test {len(id_te)} clips (sin tocar)')

## 2. Celda estructural — SOLO ClipIDs (sin etiquetas ni predicciones)

Regla pre-registrada de consecutividad: dos clips son consecutivos si comparten participante
(6 primeros dígitos) y **7.º dígito** (video/sesión), y el sufijo numérico difiere en +1.
Es la intersección conservadora de las dos interpretaciones posibles de la numeración de
DAiSEE: nunca forma una ventana que cruce participante ni video.

In [ ]:
def parse_clip(cid):
    cid = str(cid)
    return cid[:6], cid[6], int(cid[6:])          # (participante, dígito de video, sufijo entero)

def corridas(ids):
    '''Corridas máximas de clips consecutivos. Devuelve listas de índices sobre `ids`.'''
    info = sorted((*parse_clip(c), i) for i, c in enumerate(ids))
    runs, run = [], [info[0]]
    for prev, cur in zip(info, info[1:]):
        if cur[0] == prev[0] and cur[1] == prev[1] and cur[2] == prev[2] + 1:
            run.append(cur)
        else:
            runs.append(run); run = [cur]
    runs.append(run)
    return [[t[3] for t in r] for r in runs]

def ventanas_de(ids, k):
    '''Ventanas de k clips, sin solapamiento, barrido codicioso dentro de cada corrida.'''
    return [r[j:j + k] for r in corridas(ids) for j in range(0, len(r) - k + 1, k)]

for nombre, ids in [('Pool', id_pool), ('Test', id_te)]:
    rs = corridas(ids)
    largos = np.array([len(r) for r in rs])
    wins = ventanas_de(ids, K_PRIMARIO)
    # assert estructural: ninguna ventana cruza participante ni 7.º dígito
    for w in wins:
        pv = {(parse_clip(ids[i])[0], parse_clip(ids[i])[1]) for i in w}
        assert len(pv) == 1, f'Ventana cruza participante/video: {[ids[i] for i in w]}'
    cobertura = sum(len(w) for w in wins) / len(ids)
    print(f'{nombre}: {len(ids)} clips → {len(rs)} corridas '
          f'(mediana {int(np.median(largos))}, máx {largos.max()}) → '
          f'{len(wins)} ventanas k={K_PRIMARIO} (cobertura {cobertura:.1%})')
print('\nAsserts estructurales OK: ninguna ventana cruza participante ni video.')

## 3. Funciones de evaluación por ventana + bootstrap clusterizado

In [ ]:
def metricas(y, pred, prefijo=''):
    triv = max((y == 0).mean(), (y == 1).mean())
    return {f'{prefijo}n': int(len(y)),
            f'{prefijo}trivial': round(float(triv), 4),
            f'{prefijo}accuracy': round(float(accuracy_score(y, pred)), 4),
            f'{prefijo}f1_macro': round(float(f1_score(y, pred, average="macro")), 4),
            f'{prefijo}balanced_acc': round(float(balanced_accuracy_score(y, pred)), 4),
            f'{prefijo}brecha': round(float(accuracy_score(y, pred) - triv), 4)}

def agregar_ventanas(ids, y_clip, p_clip, k, umbral=UMBRAL, modo='media'):
    '''Devuelve (y_w, pred_w, p_w, part_w, homog_w) para ventanas de k clips.'''
    wins = ventanas_de(ids, k)
    y_w, pred_w, p_w, part_w, homog = [], [], [], [], []
    for w in wins:
        yy, pp = y_clip[w], p_clip[w]
        y_w.append(int(yy.sum() * 2 > len(w)))              # mayoría (k impar: sin empates)
        if modo == 'media':
            pw = float(pp.mean()); p_w.append(pw); pred_w.append(int(pw > umbral))
        else:                                                # voto duro (secundario)
            pred_w.append(int((pp > umbral).sum() * 2 > len(w))); p_w.append(float(pp.mean()))
        part_w.append(parse_clip(ids[w[0]])[0])
        homog.append(int(yy.min() == yy.max()))
    return (np.array(y_w), np.array(pred_w), np.array(p_w),
            np.array(part_w), np.array(homog, dtype=bool))

def bootstrap_cluster(part_w, y_w, pred_w, part_c, y_c, pred_c,
                      n_boot=N_BOOT, seed=SEED):
    '''Remuestreo de PARTICIPANTES con reemplazo (regla 6: IC o no existe).
    ICs para (accuracy ventana − trivial ventana) y (accuracy ventana − accuracy clip).'''
    rng = np.random.default_rng(seed)
    parts = np.unique(part_c)
    iw = {p: np.where(part_w == p)[0] for p in parts}
    ic = {p: np.where(part_c == p)[0] for p in parts}
    a = np.empty((n_boot, 2))
    for b in range(n_boot):
        sel = rng.choice(parts, size=len(parts), replace=True)
        jw = np.concatenate([iw[p] for p in sel if len(iw[p])])
        jc = np.concatenate([ic[p] for p in sel])
        acc_w = float((y_w[jw] == pred_w[jw]).mean())
        triv_w = max(float((y_w[jw] == 0).mean()), float((y_w[jw] == 1).mean()))
        acc_c = float((y_c[jc] == pred_c[jc]).mean())
        a[b] = (acc_w - triv_w, acc_w - acc_c)
    pct = lambda col: [round(v, 4) for v in np.percentile(a[:, col], [2.5, 97.5])]
    return {'IC95_brecha_ventana': pct(0), 'IC95_dif_ventana_vs_clip': pct(1)}

# ---- Preprocesamiento idéntico a 3.3 (para reajustar por fold en la CV) ----
def interpolate_sequence(seq):
    df = pd.DataFrame(seq).interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean(X, medians=None, thresh=0.30):
    keep, out = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > thresh:
            continue
        s = interpolate_sequence(X[i])
        if np.isnan(s).any() and medians is not None:
            idx = np.where(np.isnan(s)); s[idx] = np.take(medians, idx[1])
        out.append(s); keep.append(i)
    return np.stack(out), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X):
    d_ = np.zeros_like(X[:, :, delta_idx]); d_[:, 1:, :] = np.diff(X[:, :, delta_idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

def resumir(X):
    X = np.nan_to_num(X)
    return np.hstack([X.mean(axis=1), X.std(axis=1)])

print('Funciones listas.')

## 4. DESARROLLO — CV interna de 5 folds por participante

El modelo A es una regresión logística (convexa, determinística): acá la variabilidad viene
de los **folds**, no de semillas — sustituto declarado del esquema de 5 semillas (§4 del
protocolo). Preprocesamiento ajustado SOLO con el train de cada fold. Assert de leakage en
cada fold (regla 7).

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
filas, filas_k, filas_sens = [], [], []

for fold, (tr, va) in enumerate(sgkf.split(Xg_pool, y_pool, groups=part_pool)):
    assert not (set(part_pool[tr]) & set(part_pool[va])), f'LEAKAGE fold {fold}'   # regla 7

    g_med = np.nanmedian(Xg_pool[tr].reshape(-1, 22), axis=0)
    e_med = np.nanmedian(Xe_pool[tr].reshape(-1, EMB_DIM), axis=0)
    Xg_a, kga = clean(Xg_pool[tr], g_med);  Xg_b, kgb = clean(Xg_pool[va], g_med)
    Xe_a, kea = clean(Xe_pool[tr], e_med, thresh=0.5); Xe_b, keb = clean(Xe_pool[va], e_med, thresh=0.5)
    ka = np.intersect1d(kga, kea); kb = np.intersect1d(kgb, keb)
    pos = lambda kf: {k: i for i, k in enumerate(kf)}
    ga, ea = pos(kga), pos(kea); gb, eb = pos(kgb), pos(keb)

    gsc = StandardScaler().fit(add_deltas(Xg_a).reshape(-1, 32))
    esc = StandardScaler().fit(Xe_a.reshape(-1, EMB_DIM))
    sg = lambda X: gsc.transform(X.reshape(-1, 32)).reshape(X.shape[0], N_FRAMES, 32)
    se = lambda X: esc.transform(X.reshape(-1, EMB_DIM)).reshape(X.shape[0], N_EMB_FRAMES, EMB_DIM)

    R_tr = np.hstack([resumir(sg(add_deltas(Xg_a))[[ga[k] for k in ka]]),
                      resumir(se(Xe_a)[[ea[k] for k in ka]])])
    R_va = np.hstack([resumir(sg(add_deltas(Xg_b))[[gb[k] for k in kb]]),
                      resumir(se(Xe_b)[[eb[k] for k in kb]])])
    y_tr_f, y_va_f = y_pool[tr][ka], y_pool[va][kb]
    ids_va_f = id_pool[va][kb]

    rsc = StandardScaler().fit(R_tr)
    lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)
    lr.fit(rsc.transform(R_tr), y_tr_f)
    p_va = lr.predict_proba(rsc.transform(R_va))[:, 1]
    pred_c = (p_va > UMBRAL).astype(int)

    fila = {'fold': fold, **metricas(y_va_f, pred_c, 'clip_')}
    y_w, pred_w, p_w, part_w, homog = agregar_ventanas(ids_va_f, y_va_f, p_va, K_PRIMARIO)
    fila.update(metricas(y_w, pred_w, 'vent_'))
    filas.append(fila)

    # ---- Secundarios pre-registrados (solo CV) ----
    for k in KS_SECUNDARIOS:
        yw, pw_, _, _, _ = agregar_ventanas(ids_va_f, y_va_f, p_va, k)
        filas_k.append({'fold': fold, 'k': k, **metricas(yw, pw_)})
    yw, pw_, _, _, _ = agregar_ventanas(ids_va_f, y_va_f, p_va, K_PRIMARIO, modo='duro')
    filas_sens.append({'fold': fold, 'variante': 'voto_duro', **metricas(yw, pw_)})
    filas_sens.append({'fold': fold, 'variante': 'homogeneas',
                       **metricas(y_w[homog], pred_w[homog])})
    print(f'fold {fold}: clip acc {fila["clip_accuracy"]:.4f} (triv {fila["clip_trivial"]:.4f}) | '
          f'ventana acc {fila["vent_accuracy"]:.4f} (triv {fila["vent_trivial"]:.4f}, '
          f'n={fila["vent_n"]})')

df_cv = pd.DataFrame(filas)
print('\n=== CV interna, k=5 — media ± σ entre folds ===')
for c in ['clip_accuracy', 'clip_brecha', 'vent_accuracy', 'vent_f1_macro',
          'vent_balanced_acc', 'vent_trivial', 'vent_brecha']:
    print(f'  {c:18s} {df_cv[c].mean():.4f} ± {df_cv[c].std():.4f}')
df_cv.to_csv(f'{OUT_DIR}/cv_k5_por_fold.csv', index=False)
pd.DataFrame(filas_k).to_csv(f'{OUT_DIR}/cv_curva_k.csv', index=False)
pd.DataFrame(filas_sens).to_csv(f'{OUT_DIR}/cv_sensibilidad.csv', index=False)

print('\n=== Curva de k (secundaria, solo CV) ===')
print(pd.DataFrame(filas_k).groupby('k')[['accuracy', 'trivial', 'brecha', 'n']]
      .mean().round(4).to_string())
print('\n=== Sensibilidad (secundaria, solo CV) ===')
print(pd.DataFrame(filas_sens).groupby('variante')[['accuracy', 'trivial', 'brecha', 'n']]
      .mean().round(4).to_string())

## 5. COMPUERTA — Test oficial, UNA sola pasada

**No cambiar el flag hasta que:** (1) la CV de arriba esté corrida y registrada, (2) no se
quiera modificar nada más del protocolo. Al pasar `EJECUTAR_TEST_OFICIAL = True` se consume
la única evaluación sobre Test de este experimento (regla 1). Los análisis secundarios
NO se corren sobre Test.

In [ ]:
EJECUTAR_TEST_OFICIAL = False   # ← cambiar a True SOLO para la pasada única

In [ ]:
if not EJECUTAR_TEST_OFICIAL:
    print('Compuerta cerrada. Correr la CV, registrar, y recién entonces habilitar.')
else:
    import tensorflow as tf
    import keras
    from keras import layers

    # ---- Custom layers para poder cargar los .keras entregados ----
    @keras.saving.register_keras_serializable(package='sparkle')
    class ResumenTemporal(layers.Layer):
        def call(self, x):
            return tf.concat([tf.reduce_mean(x, axis=1), tf.math.reduce_std(x, axis=1)], axis=-1)

    @keras.saving.register_keras_serializable(package='sparkle')
    class AttentionPooling(layers.Layer):
        def __init__(self, **kw):
            super().__init__(**kw); self.score_dense = layers.Dense(1, name='attn_score')
        def call(self, x):
            w = tf.nn.softmax(self.score_dense(x), axis=1)
            return tf.reduce_sum(x * w, axis=1), tf.squeeze(w, axis=-1)

    @keras.saving.register_keras_serializable(package='sparkle')
    class CoralHead(layers.Layer):
        def __init__(self, num_classes, **kw):
            super().__init__(**kw); self.num_classes = num_classes
            self.dense = layers.Dense(num_classes - 1, name='coral_thresholds')
        def call(self, x): return tf.nn.sigmoid(self.dense(x))
        def compute_output_shape(self, s): return (s[0], self.num_classes - 1)
        def get_config(self):
            c = super().get_config(); c.update({'num_classes': self.num_classes}); return c

    @keras.saving.register_keras_serializable(package='sparkle')
    class InterpolarTemporal(layers.Layer):
        def __init__(self, n_out, **kw):
            super().__init__(**kw); self.n_out = n_out
        def call(self, x):
            y = tf.image.resize(tf.expand_dims(x, axis=2), [self.n_out, 1], method='bilinear')
            return tf.squeeze(y, axis=2)
        def compute_output_shape(self, s): return (s[0], self.n_out, s[2])
        def get_config(self):
            c = super().get_config(); c.update({'n_out': self.n_out}); return c

    # ---- Test con el preprocesamiento ENTREGADO (no se reajusta nada) ----
    prep = np.load(f'{MODELO_DIR}/preprocesamiento.npz', allow_pickle=True)
    Xg_c, kgc = clean(Xg_te, prep['geom_medians'])
    Xe_c, kec = clean(Xe_te, prep['emb_medians'], thresh=0.5)
    kc = np.intersect1d(kgc, kec)
    pg = {k: i for i, k in enumerate(kgc)}; pe = {k: i for i, k in enumerate(kec)}
    Xg_TE = add_deltas(Xg_c)[[pg[k] for k in kc]]
    Xg_TE = ((Xg_TE.reshape(-1, 32) - prep['geom_scaler_mean']) /
             prep['geom_scaler_scale']).reshape(-1, N_FRAMES, 32).astype(np.float32)
    Xe_TE = Xe_c[[pe[k] for k in kc]]
    Xe_TE = ((Xe_TE.reshape(-1, EMB_DIM) - prep['emb_scaler_mean']) /
             prep['emb_scaler_scale']).reshape(-1, N_EMB_FRAMES, EMB_DIM).astype(np.float32)
    y_TE = (y3_te > 0).astype(np.int64)[kc]
    ids_TE = id_te[kc]
    part_TE = np.array([c[:6] for c in ids_TE])
    print(f'Test reconstruido: {len(y_TE)} clips, trivial clip = '
          f'{max((y_TE==0).mean(), (y_TE==1).mean()):.4f}')

    # ---- Modelo A (primario) ----
    modelo_A = keras.models.load_model(f'{MODELO_DIR}/modelo_A_resumen_lineal.keras')
    p_A = modelo_A.predict({'geometricas': Xg_TE, 'embeddings_cnn': Xe_TE},
                           batch_size=256, verbose=0).ravel()
    pred_A = (p_A > UMBRAL).astype(int)
    acc_clip = accuracy_score(y_TE, pred_A)
    # Verificación de reproducción del número YA publicado (no es una mirada nueva):
    assert abs(acc_clip - 0.6374) < 0.002, (
        f'La accuracy por clip ({acc_clip:.4f}) no reproduce el 0,6374 publicado: '
        'el preprocesamiento no coincide con el de 3.3. NO seguir; revisar antes de usar nada.')

    res_clip = metricas(y_TE, pred_A, 'clip_')
    y_w, pred_w, p_w, part_w, homog = agregar_ventanas(ids_TE, y_TE, p_A, K_PRIMARIO)
    res_vent = metricas(y_w, pred_w, 'vent_')
    ics = bootstrap_cluster(part_w, y_w, pred_w, part_TE, y_TE, pred_A)

    print('\n================ TEST OFICIAL — MODELO A (k=5) ================')
    print(f'Por clip    : acc {res_clip["clip_accuracy"]:.4f} | trivial {res_clip["clip_trivial"]:.4f} '
          f'| brecha {res_clip["clip_brecha"]:+.4f} | F1M {res_clip["clip_f1_macro"]:.4f} '
          f'| BalAcc {res_clip["clip_balanced_acc"]:.4f} | n={res_clip["clip_n"]}')
    print(f'Por ventana : acc {res_vent["vent_accuracy"]:.4f} | trivial {res_vent["vent_trivial"]:.4f} '
          f'| brecha {res_vent["vent_brecha"]:+.4f} | F1M {res_vent["vent_f1_macro"]:.4f} '
          f'| BalAcc {res_vent["vent_balanced_acc"]:.4f} | n={res_vent["vent_n"]}')
    print(f'IC95 brecha ventana (bootstrap por participante): {ics["IC95_brecha_ventana"]}')
    print(f'IC95 (acc ventana − acc clip):                    {ics["IC95_dif_ventana_vs_clip"]}')
    if ics['IC95_dif_ventana_vs_clip'][0] <= 0 <= ics['IC95_dif_ventana_vs_clip'][1]:
        print('⚠️  El IC de la diferencia cruza cero: la mejora por agregación NO se afirma (regla 6).')
    if ics['IC95_brecha_ventana'][0] <= 0:
        print('⚠️  El IC de la brecha por ventana toca cero: el modelo no supera al trivial en ventanas.')

    # ---- Modelo B (secundario, descriptivo del artefacto desplegado) ----
    res_B = None
    ruta_B = f'{MODELO_DIR}/modelo_B_tcn_fusion.keras'
    if os.path.exists(ruta_B):
        modelo_B = keras.models.load_model(ruta_B)
        outs = modelo_B.predict({'embeddings_cnn': Xe_TE, 'geometricas': Xg_TE},
                                batch_size=256, verbose=0)
        outs = outs if isinstance(outs, (list, tuple)) else [outs]
        p_B = next(o for o in outs if o.ndim == 2 and o.shape[1] == 1).ravel()  # por FORMA, no índice
        pred_Bc = (p_B > UMBRAL).astype(int)
        yBw, predBw, _, partBw, _ = agregar_ventanas(ids_TE, y_TE, p_B, K_PRIMARIO)
        res_B = {'clip': metricas(y_TE, pred_Bc), 'ventana': metricas(yBw, predBw),
                 'ics': bootstrap_cluster(partBw, yBw, predBw, part_TE, y_TE, pred_Bc)}
        print('\n----- Modelo B (secundario, 1 artefacto — NO comparar arquitecturas) -----')
        print(f'Por clip    : acc {res_B["clip"]["accuracy"]:.4f} (triv {res_B["clip"]["trivial"]:.4f})')
        print(f'Por ventana : acc {res_B["ventana"]["accuracy"]:.4f} '
              f'(triv {res_B["ventana"]["trivial"]:.4f}, n={res_B["ventana"]["n"]})')
        print(f'ICs: {res_B["ics"]}')

    # ---- Persistencia ----
    resultado = {'protocolo': 'Protocolo_Exp1_Agregacion_Temporal.md',
                 'fecha': pd.Timestamp.now().isoformat(), 'k': K_PRIMARIO,
                 'umbral': UMBRAL, 'n_boot': N_BOOT,
                 'modelo_A': {**res_clip, **res_vent, **ics},
                 'modelo_B': res_B,
                 'evaluaciones_test_este_experimento': 1}
    with open(f'{OUT_DIR}/test_oficial_resultado.json', 'w') as f:
        json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)
    print(f'\nGuardado en {OUT_DIR}/test_oficial_resultado.json')
    print('Registrar en Fase3_Registro_de_Resultados.md — éxito o fracaso, mismo detalle (regla 10).')

## 6. Después de correr

1. Pegar los números de la CV **y** del Test (si se habilitó) en
   `Fase3_Registro_de_Resultados.md`, con este bloque como plantilla:
   - protocolo pre-registrado y fecha · k=5 · mayoría · media de probabilidades · umbral 0,50
   - tabla CV por fold (clip y ventana) con media ± σ
   - Test: clip y ventana lado a lado, ambos triviales, ambas brechas, F1 macro, balanced acc.
   - los dos IC bootstrap y su lectura (si cruzan cero, la diferencia no existe)
   - contador: evaluaciones sobre Test de este experimento = 1
2. Si el resultado por ventana llega a 70 %: verificar antes de celebrar que la diferencia
   ventana−clip no cruza cero y que el trivial por ventana no subió tanto como la accuracy
   (si el trivial por ventana es ~64 %, un 70 % de accuracy sería una brecha de solo 6 pts).
   **El número que importa para el jurado es la brecha, no la accuracy.**
3. Los secundarios (curva de k, voto duro, homogéneas) quedan como material de CV interna.
   No se corren sobre Test.